# Task 2 — The Mystery Black Box
### Testing: Linear Transformations

## Assignment scenario

You are handed an undocumented legacy function that transforms 2D image
coordinates. You have mapped its behavior based on two basis vectors:

- Inputting `(1, 0)` outputs `(0, 1)`
- Inputting `(0, 1)` outputs `(-1, 0)`

**Objective:** deduce the transformation matrix governing this black box,
implement raw matrix-vector multiplication from scratch, and observe the
sequential effects of **compounding** (repeatedly applying) that
transformation.

**Constraint:** no NumPy, no Pandas. All matrix/vector operations come
from our own `src/linalg.py`.

## Step 0 — Why two probes are enough

A **linear transformation** `T` on R^2 is completely determined by what it
does to the two standard basis vectors, `e1 = (1, 0)` and `e2 = (0, 1)`.
Any point `(x, y)` can be written as `x*e1 + y*e2`, and linearity means:

```
T(x, y) = x * T(e1) + y * T(e2)
```

So knowing `T(e1)` and `T(e2)` is knowing all of `T`. In matrix form: **the
columns of the transformation matrix are exactly `T(e1)` and `T(e2)`.**
That's the entire trick behind this task.

In [ ]:
import sys
sys.path.insert(0, ".")
from linalg import matmul, mat_vec_mul, apply_transformation

from secret_box import black_box  # the only thing we're "allowed" to call


## Step 1 — Read off the reported probes

The assignment already reports what the black box does to `e1` and `e2` —
we don't even need to probe it ourselves, just trust (and then double
check against) the reported behavior.

In [ ]:
e1 = (1, 0)
e2 = (0, 1)

col1 = black_box(e1)
col2 = black_box(e2)

print("black_box(1, 0) =", col1, " (assignment reports (0, 1))")
print("black_box(0, 1) =", col2, " (assignment reports (-1, 0))")

assert col1 == (0, 1), "black_box output for e1 doesn't match the assignment"
assert col2 == (-1, 0), "black_box output for e2 doesn't match the assignment"
print()
print("Confirmed: black_box matches the reported behavior.")


## Step 2 — Deduce the transformation matrix

Since `T(e1)` gives column 1 and `T(e2)` gives column 2, we can build the
full matrix directly, with zero guessing:

In [ ]:
deduced_matrix = [
    [col1[0], col2[0]],
    [col1[1], col2[1]],
]

print("Deduced matrix M:")
for row in deduced_matrix:
    print(row)


## Step 3 — Recognize the transformation

Look closely at the deduced matrix:

```
M = [ 0  -1 ]
    [ 1   0 ]
```

This is exactly the standard **90-degree counter-clockwise rotation
matrix**. Sanity check: rotating `(1, 0)` (pointing along the x-axis) by
90 degrees counter-clockwise should point along the y-axis, i.e. land on
`(0, 1)` — which matches `T(e1)` exactly. Same logic confirms `T(e2)`.

In [ ]:
import math

def rotation_matrix(theta_degrees):
    theta = math.radians(theta_degrees)
    c, s = math.cos(theta), math.sin(theta)
    return [[round(c, 10), round(-s, 10)], [round(s, 10), round(c, 10)]]

expected_90 = rotation_matrix(90)
print("Standard 90-degree rotation matrix:")
for row in expected_90:
    print(row)

matches = all(
    abs(deduced_matrix[i][j] - expected_90[i][j]) < 1e-9
    for i in range(2) for j in range(2)
)
print()
print("Deduced matrix matches a 90-degree rotation?", matches)


## Step 4 — Verify raw matrix-vector multiplication on new points

A matrix deduced from just 2 probes is only useful if it correctly
predicts *every other* input. Let's implement and test raw
matrix-vector multiplication (from `linalg.py`, no NumPy) against fresh
points the black box hasn't been asked about yet.

In [ ]:
test_points = [(2, 3), (-1, 4), (5, -2), (3, 3), (-2, -2)]

print(f"{'point':>10} | {'black_box':>12} | {'M @ point (raw mat-vec mult)':>28} | match?")
all_match = True
for p in test_points:
    actual = black_box(p)
    predicted = mat_vec_mul(deduced_matrix, p)
    match = actual == predicted
    all_match &= match
    print(f"{str(p):>10} | {str(actual):>12} | {str(predicted):>28} | {match}")

print()
print("All predictions matched!" if all_match else "Mismatch found -- something's wrong.")


## Step 5 — Compounding: observe sequential effects

The assignment specifically asks us to "observe the sequential effects of
compounding transformations." A 90-degree rotation is the perfect
demonstration: applying it once rotates a point 90 degrees, applying it
**twice in sequence** should rotate it 180 degrees total, three times =
270 degrees, and four times should bring the point all the way back to
where it started (360 degrees = identity).

We implement this by repeatedly feeding a point back through
`black_box()` -- i.e. literally composing the transformation with itself,
step by step -- rather than jumping straight to a closed-form matrix
power.

In [ ]:
def compound(point, times):
    """Feed `point` through black_box() `times` times in sequence,
    printing each intermediate result so the compounding effect is visible
    step by step."""
    current = point
    print(f"Start:      {current}")
    for step in range(1, times + 1):
        current = black_box(current)
        degrees = 90 * step
        print(f"After {step} application(s) ({degrees:>3} deg total): {current}")
    return current


start_point = (1, 0)
final = compound(start_point, 4)
print()
print("Back to start after 4 applications (360 degrees)?", final == start_point)


## Step 6 — Compounding as matrix multiplication (order matters, in general)

Repeatedly applying `black_box` is mathematically the same as repeatedly
multiplying by the deduced matrix `M`. Applying it `k` times corresponds
to computing `M^k` (M multiplied by itself `k` times) and applying that
once. We verify this equivalence directly using our own `matmul`.

In [ ]:
def matrix_power(M, k):
    """Raw matrix power using our own matmul -- no NumPy."""
    n = len(M)
    result = [[1 if i == j else 0 for j in range(n)] for i in range(n)]
    for _ in range(k):
        result = matmul(result, M)
    return result

for k in range(1, 5):
    Mk = matrix_power(deduced_matrix, k)
    via_matrix_power = mat_vec_mul(Mk, start_point)

    # Recompute via k repeated black_box() calls, for comparison
    p = start_point
    for _ in range(k):
        p = black_box(p)

    print(f"k={k}: M^{k} @ {start_point} = {via_matrix_power}   |   "
          f"{k}x black_box({start_point}) = {p}   |   match: {via_matrix_power == p}")


## Step 7 — Exercises for interns

1. **Why 4 applications = identity.** Explain, using the rotation angle,
   why compounding this specific black box exactly 4 times always returns
   any point to its original position, regardless of which point you
   start with.
2. **A different black box.** Suppose a new black box reports
   `(1,0) -> (2, 0)` and `(0,1) -> (0, 0.5)`. Deduce its matrix, and
   figure out how many times you'd need to compound it before a generic
   point returns to (approximately) its original position -- or explain
   why it never will.
3. **Order matters, generally.** This task's rotation matrix happens to
   commute with itself (any matrix trivially commutes with its own
   powers). Construct two *different* 2x2 matrices `A` and `B` (e.g. a
   rotation and a shear) and show with `matmul` that `A @ B != B @ A` in
   general -- reinforcing why "compounding" a sequence of *different*
   transformations is order-sensitive, even though compounding the *same*
   one repeatedly is not.
4. **Affine trap.** Everything here assumes the black box is purely
   linear (no added constant term). What single extra probe would reveal
   whether the box is secretly *affine* instead (`T(x) = Mx + b`)? Hint:
   what must a truly linear transformation always do to `(0, 0)`?